# Context Routing & Intelligent Selection
## Selecting the Best Context from Multiple Candidates

In this notebook, we'll build an intelligent context routing system that:
- **Retrieves** candidate documents using semantic search
- **Ranks** by multiple factors (similarity, priority, recency)
- **Resolves** topic conflicts to avoid redundant context
- **Routes** queries to the most relevant context
- **Generates** grounded answers using selected context

This is the capstone of Day 1, tying together all previous segments into an intelligent system.

## Step 1: Setup & Configuration

Initialize environment variables and import core date/time utilities for timestamp-based ranking.

In [12]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

## Step 2: Import Libraries

Import core components for document handling and time-based ranking.

In [13]:
from datetime import datetime, timedelta
from langchain_core.documents import Document

## Step 3: Sample Documents with Rich Metadata

Create documents with multiple metadata dimensions for intelligent ranking:

**Metadata Fields:**
- **topic**: Categorization (product, finance, etc.) - for conflict resolution
- **source_type**: Origin of document (notes, report, experiment) - for trust scoring
- **priority**: Importance ranking (1-5) - for weighted scoring
- **timestamp**: Creation date - for recency calculation

This rich metadata enables sophisticated routing decisions.

In [25]:
documents = [
    Document(
        page_content="Subscription pricing is preferred by customers.",
        metadata={
            "topic": "product",
            "source_type": "notes",
            "priority": 2,
            "timestamp": (datetime.now() - timedelta(days=1)).isoformat()
        }
    ),
    Document(
        page_content="Revenue increased by 30% after switching to tiered pricing.",
        metadata={
            "topic": "finance",
            "source_type": "report",
            "priority": 3,
            "timestamp": (datetime.now() - timedelta(days=10)).isoformat()
        }
    ),
    Document(
        page_content="Freemium model failed in enterprise segment.",
        metadata={
            "topic": "product",
            "source_type": "experiment",
            "priority": 5,
            "timestamp": (datetime.now() - timedelta(days=2)).isoformat()
        }
    )
]

## Step 4: Build Vector Store

Create a Chroma vector store with all documents, storing their embeddings and metadata for retrieval.

In [26]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small",
                                openai_api_key=os.getenv("OPENAI_API_KEY"))

vector_store = Chroma.from_documents(
    documents,
    embedding=embeddings,
    collection_name="router_demo"
)

## Step 5: Retrieve Candidates

Perform semantic similarity search to get candidate documents with their similarity scores.

**Key Point:** We retrieve k=5 candidates to have options before ranking and filtering.

In [16]:
def retrieve_candidates(query):
    return vector_store.similarity_search_with_score(query, k=5)

## Step 6: Multi-Factor Scoring

Rank each document using three independent signals:

**Scoring Factors:**
1. **Similarity (50% weight)**: How semantically relevant is the document?
   - Formula: 1/(1+distance) - closer vectors = higher score
   
2. **Priority (30% weight)**: How important is this document?
   - Direct metadata field (1-5 scale)
   
3. **Recency (20% weight)**: How fresh is the information?
   - Formula: 1/(1+days_old) - newer documents score higher

**Weighted Formula:**
```
final_score = (0.5 × similarity) + (0.3 × priority) + (0.2 × recency)
```

This balances semantic relevance, importance, and freshness.

In [17]:
from datetime import datetime

def compute_score(doc, similarity_score):

    # Normalize similarity (lower is better in Chroma → convert)
    similarity = 1 / (1 + similarity_score)

    # Priority (higher = more important)
    priority = doc.metadata.get("priority", 1)

    # Recency (more recent = higher score)
    timestamp_str = doc.metadata.get("timestamp")
    if timestamp_str:
        timestamp = datetime.fromisoformat(timestamp_str)
    else:
        timestamp = datetime.now()
    age_days = (datetime.now() - timestamp).days
    recency = 1 / (1 + age_days)

    # Final weighted score
    final_score = (
        0.5 * similarity +
        0.3 * priority +
        0.2 * recency
    )

    return final_score

## Step 7: Rank by Composite Score

Sort all candidates by their combined scores in descending order (highest score first).

In [18]:
def rank_contexts(results):

    scored = []

    for doc, sim_score in results:
        score = compute_score(doc, sim_score)
        scored.append((doc, score))

    # Sort descending
    scored.sort(key=lambda x: x[1], reverse=True)

    return scored

## Step 8: Resolve Topic Conflicts

Avoid redundant context by keeping only the highest-scoring document per topic.

**Why?**
- Multiple documents about the same topic may present conflicting viewpoints
- Using the best-ranked document per topic ensures diversity
- Reduces context window usage while maintaining coverage
- Prevents bias toward any single topic

**Process:**
1. Iterate through ranked documents
2. Track topics already selected
3. Skip documents from already-selected topics
4. Keep first (best-scored) document per unique topic

In [19]:
def resolve_conflicts(scored_docs):

    selected = []
    seen_topics = set()

    for doc, score in scored_docs:

        topic = doc.metadata.get("topic")

        # If already have context for this topic, skip weaker ones
        if topic in seen_topics:
            continue

        selected.append(doc)
        seen_topics.add(topic)

    return selected

## Step 9: Context Router (Complete Pipeline)

Orchestrate all steps into a single intelligent routing function:

**The Context Routing Pipeline:**
```
Query
  ↓
[Step 1: Retrieve] → Get top-k semantic candidates
  ↓
[Step 2: Rank] → Score by similarity + priority + recency
  ↓
[Step 3: Resolve] → One best document per topic
  ↓
Final Selected Context
```

This is the intelligent heart of the system - deciding what context to use.

In [20]:
def context_router(query):

    # Step 1: Retrieve
    results = retrieve_candidates(query)

    # Step 2: Rank
    ranked = rank_contexts(results)

    # Step 3: Resolve conflicts
    final_context = resolve_conflicts(ranked)

    return final_context

## Step 10: Context-Augmented Generation

Complete the RAG loop by combining routed context with LLM generation:

**The Full RAG System:**
```
User Query
  ↓
[Context Router] → Select best context (Steps 1-9)
  ↓
[Build Prompt] → Combine system message + context + question
  ↓
[LLM Generation] → Generate answer grounded in context
  ↓
Informed Response
```

This demonstrates the full power of context engineering:
- **Intelligent retrieval** picks the most relevant documents
- **Multi-factor ranking** balances competing priorities
- **Conflict resolution** ensures diversity and clarity
- **LLM augmentation** produces grounded, reliable answers

In [21]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o", temperature=0)

def generate_answer(query):

    contexts = context_router(query)

    combined_context = "\n\n".join(
        [doc.page_content for doc in contexts]
    )

    messages = [
        SystemMessage(content="You are a product strategist."),
        HumanMessage(content=f"""
Context:
{combined_context}

Question:
{query}
""")
    ]

    response = llm.invoke(messages)

    return response.content

## Step 11: Execute End-to-End System

Run a complete query through the entire context engineering pipeline and generate an informed answer.

In [22]:
query = "What pricing strategy should we adopt?"

answer = generate_answer(query)

print(answer)

Given the context that the freemium model failed in the enterprise segment and revenue increased by 30% after switching to tiered pricing, it seems that the tiered pricing strategy is more effective for your enterprise customers. Here are some considerations and recommendations for adopting a pricing strategy:

1. **Continue with Tiered Pricing:**
   - Since the tiered pricing model has already shown a positive impact on revenue, it would be wise to continue with this approach. It allows you to cater to different segments of the enterprise market by offering various levels of service at different price points.

2. **Optimize Tier Levels:**
   - Analyze customer data to understand which tiers are most popular and why. Consider adjusting the features or pricing of each tier to better align with customer needs and maximize revenue.

3. **Value-Based Pricing:**
   - Ensure that each tier is priced based on the value it provides to the customer. This involves understanding the specific need

## Key Takeaways: The Complete Context Engineering System

### The Three-Step Selection Pattern

Context Engineering follows an intelligent three-step selection process:

```
Candidates (Many)
    ↓
[Retrieve] ← Semantic similarity search
    ↓
Ranked List
    ↓
[Rank] ← Multi-factor scoring
    ↓
Filtered Set
    ↓
[Resolve] ← Conflict resolution
    ↓
Final Context (Few, High Quality)
    ↓
LLM Generation
```

### Why Multi-Factor Ranking?

Real decisions require balancing competing priorities:

| Factor | Weight | Why | Example |
|--------|--------|-----|---------|
| **Similarity** | 50% | What's relevant to this query? | Match user's question |
| **Priority** | 30% | What's most important? | Reports > casual notes |
| **Recency** | 20% | What's most current? | Q4 data > last year |

### The Power of Conflict Resolution

Avoid information overload:
- **Without resolution**: May include 3+ documents about pricing (overlapping)
- **With resolution**: One best document per topic (diverse coverage)
- **Result**: Clearer context, smaller context window, better answers

### Production Patterns

This system demonstrates patterns used in production RAG systems:

1. **Staged Filtering**: Broad → Ranked → Precise
2. **Multi-Dimensional Scoring**: Never optimize for one dimension alone
3. **Metadata-Driven Decisions**: Rich metadata enables intelligent routing
4. **Graceful Degradation**: Fallback strategies if primary methods fail
5. **Composability**: Each step is a reusable function

### What You've Learned (Day 1 Complete!)

✓ **Segment 1**: Context layering (system, domain, environment, session, prompt)  
✓ **Segment 2**: Document ingestion (load, validate, enrich)  
✓ **Segment 3**: Vector embeddings & retrieval (semantic search)  
✓ **Segment 4**: Context routing & ranking (intelligent selection)

You now have the foundation for building sophisticated RAG systems!

---

**Next Steps (Day 2):**
- Token management and context compression
- Multi-source integration
- Advanced retrieval techniques
- Production system design